# Bispecific ADC upgrades — implementation and results (v3)

Frozen v2 reconstruction → differential trafficking → recycling → linker shedding → active payload exchange. **Exploratory model experiments, not validated biological predictions.** All implementation is contained here; restart and run all. Existing v2 is unchanged. Outputs are written to `results_v3/`.

Keep Table 1 multipliers, 4 µmol / 2 h dose, 24 h endpoint, DAR 1, EC50 4 µM, geometry and v2 calibration fixed. No refitting after enabling mechanisms. Optimise four binding rates under **0.95% global healthy effect**, retaining the strict <1% criterion. “Effect” is the paper-style Emax surrogate; with exchange it is based on instantaneous active payload, not irreversible accumulated cell death.

### Saved run: main results

Default settings; modelled **24-hour Emax effects**, not validated irreversible killing. Each row is independently reoptimised; healthy effect is approximately 0.95%.

| Scenario | Tumour effect | Healthy effect |
|---|---:|---:|
| Baseline | 97.49% | 0.95% |
| 5× dual internalisation | 98.06% | 0.95% |
| Stable linker + payload exchange | 95.96% | 0.95% |
| 96 h linker + payload exchange | 95.22% | 0.95% |
| Combined mechanisms | 95.99% | 0.95% |
| 96 h linker + exchange, 11.3 L payload pool | 89.99% | 0.95% |

- **Trafficking helps modestly after retuning.** Fivefold dual uptake gains 0.57 percentage points over baseline; the unretuned antibody instead exceeds 1% healthy effect. Faster uptake also acts on H12.
- **Density changes the preferred asymmetry.** For slow T1 / fast T2 with intermediate dual uptake, equal-density KD values are 114 and 2,260 µM; at 20:1 target density they are 1,614 and 1,596 µM. The fast-target affinity rule is context-dependent.
- **Payload transport matters more than shedding alone in these scenarios.** Shedding only (96 h) retains 97.44% tumour effect after retuning. Stable-linker exchange already lowers it to 95.96%; shedding plus exchange lowers it further to 95.22%. Increasing PS/VI from 0.01 to 1 h⁻¹ at a 96 h linker shifts the best-found effect from 97.25% to 75.63%.
- **No demonstrated interior optimum or collapse.** Feasible candidates were found in every tested optimisation, and every selected candidate touches a rate bound. Neither global optimality nor engineering feasibility is established.
- **The global endpoint conceals subgroup injury.** Baseline H12 effect is 81.19%, despite only 0.95% globally. Primary selected designs peak below 1% on the sampled time grid (maximum 0.9643%); this does not validate clinical safety.
- **New PK parameters require measurements.** Reducing the assumed payload pool from 52.8 to 11.3 L lowers reoptimised tumour effect from 95.22% to 89.99%. These volumes and other new rates are exploratory, not confidence bounds.

The 24 primary designs passed mass balance and tolerance checks; maximum relative mass error was 1.17×10⁻¹². Independent solver, v2 recovery, target-swap and intact/naked invariance tests passed. Two-seed tumour-effect disagreement was at most 0.00149 percentage points. Small seed disagreement does not prove unique or global optimality. The tables below retain all rates, subgroup effects and search records.


### 1. Frozen controls and declared new assumptions

`NCELL` and `VI` are the full-precision v2 calibration (7.044×10¹² cells, 0.268972 pL/cell). Table 1 multipliers sum to 1.0003; healthy weights sum to 1. Bound turnover varies separately from free-receptor loss; synthesis remains `k0 × rho`. Drug-free free-receptor turnover is an effective loss, with no separate free-receptor recycling pool.

New rates are **scenario assumptions**, not fitted measurements. `PS/VI` = 0.01, 0.1, 1 h⁻¹; payload clearance/volume = 0.01, 0.1, 1 h⁻¹; linker half-life = 48, 96, 168 h. One well-mixed **pooled extracellular payload** compartment receives cleavage from both antibody compartments and surfaces. Its exploratory volume is 52.8 L; instantaneous pooling is a structural approximation. This explicitly accounts for peripheral cleavage without claiming that peripheral payload physically appears immediately in central plasma.

With endosomes enabled, intact complexes recycle, process to naked complexes + payload, or degrade + release payload. Naked complexes recycle or degrade. Attached receptors follow their complex and are destroyed on degradation; return preserves binding configuration. `tau` is the mean time to an intact state's first exit, not naked-antibody residence time. Processing/degradation split is 50:50. Without endosomes, internalisation immediately degrades antibody and releases one payload. Background intact-ADC clearance removes its payload from the model.

Primary binding closure is v2 symmetric cycle closure. The alternative is explicitly **uncoupled** (`q_i=koff_i`); v2's option named `sengers` is different. Slow T1 / fast T2 labels avoid the planning chats' reversal of guide/effector terminology in Lee (2019).

Extracellular cleavage and intracellular processing are assumed to generate the same immediately active, permeable payload; chemistry-specific inactive or poorly permeable catabolites are not resolved. Linker half-life applies to circulating/surface intact ADC; endosomal processing is specified separately.

In [ ]:
get_ipython().run_line_magic('matplotlib','inline')
from dataclasses import dataclass, replace, asdict
from pathlib import Path
import itertools, json, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import minimize
from platypus import Problem, Real, Solution, InjectedPopulation, NSGAII
from IPython.display import display, Markdown

# Hours; solution and intracellular concentrations in uM; amounts in umol;
# surface/endosomal states in units of 1e-10 mol/dm2.
F = np.array([.0003, .79, .10, .10, .01])
NAMES = ['T', 'H0', 'H1', 'H2', 'H12']
NCELL = 7043996587630.85
VI = 2.689720494498626e-13
AREA = 3.141592653589793e-8
CAP = NCELL * F * AREA * 1e-4  # umol per scaled surface unit
CELLVOL = NCELL * F * VI     # L
GAIN = AREA * 1e-4 / VI      # uM per scaled surface unit
VC, VP, QA = 11.3, 41.5, 1.05
CLA = 0.015540799881601949
BOUNDS = [(2.,7.),(-5.,2.),(2.,7.),(-5.,2.)]
LIMIT = .0095
RATES = ['kon1','koff1','kon2','koff2']

@dataclass(frozen=True)
class Scenario:
    rho: tuple = (1., 1.)  # multiples of 1e-10 mol/dm2
    k0: tuple = (2., 2.)
    ki: tuple = (2., 2., 2.)
    half: float = np.inf
    permeability: float = 0.  # PS / VI, h^-1; bidirectional, K=fu=1
    payload_volume: float = 52.8  # one WELL-MIXED extracellular pool, exploratory
    payload_clearance: float = 5.28  # L/h; clearance rate 0.1 h^-1
    metabolism: float = 0.
    tau: float = 0.  # 0: instantaneous internal degradation and release
    recycle: float = 0.
    process_share: float = .5  # remaining exit split between processing/degradation
    closure: str = 'symmetric'
    surface_factor: float = 1.
    dose: float = 4.  # umol


def density(s):
    return np.array([[1,1],[0,0],[.25,0],[0,.25],[.25,.25]]) * s.rho




### 2. Conserved ODE implementation

All transfers use µmol/h after multiplying surface density by population surface area. Binding of intact and naked antibody shares one receptor pool. Deconjugation changes antibody identity without releasing receptors. Payload membrane flux is `PS × (P_external − P_cell)`; metabolism is zero in the primary comparisons. Historical targeted delivery `A` is diagnostic only and never used in membrane exchange or mass balance.

Antibody balance counts free + surface + endosomal + eliminated antibody. Payload balance counts intact conjugates + extracellular free + intracellular free + eliminated/inactivated payload. A dual complex carries one antibody and one payload. The solver splits at the infusion end.

In [ ]:
def make_rhs(theta, s, on):
    a1,b1,a2,b2 = 10.**np.asarray(theta) * 3600
    l = 2.4e13 * 1e-10 * 3600 * s.surface_factor
    corr = np.sqrt(a2/a1) if s.closure == 'symmetric' else 1.
    q1,q2 = b1*corr,b2/corr
    kd = np.log(2)/s.half
    rho,k0,ki = density(s), np.array(s.k0), np.array(s.ki)
    kr = s.recycle/s.tau if s.tau else 0.
    kp = (1-s.recycle)*s.process_share/s.tau if s.tau else 0.
    kg = (1-s.recycle)*(1-s.process_share)/s.tau if s.tau else 0.
    def fun(t,y):
        # 0:4 Ic,Ip,Nc,Np; 4 extracellular payload; 5 Ab lost; 6 payload lost.
        # per cell: R(2), intact S(3), naked S(3), intact E(3), naked E(3),
        # P instantaneous, A cumulative productive targeted release (both uM).
        z=y[7:].reshape(5,16); dz=np.zeros_like(z); out=np.zeros_like(y)
        R=z[:,:2]; S=z[:,2:8].reshape(5,2,3); E=z[:,8:14].reshape(5,2,3)
        dS=dz[:,2:8].reshape(5,2,3); dE=dz[:,8:14].reshape(5,2,3)
        dz[:,:2]=k0*(rho-R)
        for x,c in enumerate([y[0],y[2]]):
            b=S[:,x,:]
            v1=a1*c*1e-6*R[:,0]-b1*b[:,0]
            v2=a2*c*1e-6*R[:,1]-b2*b[:,1]
            w1=l*b[:,1]*R[:,0]-q1*b[:,2]
            w2=l*b[:,0]*R[:,1]-q2*b[:,2]
            dz[:,0]-=v1+w1; dz[:,1]-=v2+w2
            dS[:,x,:]=np.column_stack([v1-w2,v2-w1,w1+w2])-ki*b
            cix=2*x; pix=cix+1
            flow=QA*(y[cix]-y[pix])
            out[cix]=(-CLA*y[cix]-flow-CAP@(v1+v2))/VC
            out[pix]=flow/VP
        shed=kd*S[:,0,:]
        dS[:,0,:]-=shed; dS[:,1,:]+=shed
        out[0]-=kd*y[0]; out[1]-=kd*y[1]
        out[2]+=kd*y[0]; out[3]+=kd*y[1]
        out[0]+=(s.dose/2/VC if on else 0.)
        if s.tau:
            dS[:]+=kr*E
            dE[:,0,:]=ki*S[:,0,:]-(kr+kp+kg)*E[:,0,:]
            dE[:,1,:]=ki*S[:,1,:]+kp*E[:,0,:]-(kr+kg)*E[:,1,:]
            release=(kp+kg)*E[:,0,:].sum(axis=1)
            ab_loss=kg*E.sum(axis=(1,2))
        else:
            release=(ki*S[:,0,:]).sum(axis=1)
            ab_loss=(ki*S.sum(axis=1)).sum(axis=1)
        production=GAIN*release
        exchange=s.permeability*(y[4]-z[:,14])
        dz[:,14]=production+exchange-s.metabolism*z[:,14]
        dz[:,15]=production
        # Both antibody compartments release to the explicitly pooled extracellular
        # payload volume; this is a lumping assumption, not central-only transport.
        generated=kd*(VC*y[0]+VP*y[1])+CAP@shed.sum(axis=1)
        out[4]=(generated-s.payload_clearance*y[4]-CELLVOL@exchange)/s.payload_volume
        out[5]=CLA*(y[0]+y[2])+CAP@ab_loss
        out[6]=CLA*y[0]+s.payload_clearance*y[4]+s.metabolism*(CELLVOL@z[:,14])
        out[7:]=dz.ravel()
        return out
    return fun


def simulate(theta, s=Scenario(), points=97, tight=False, method='BDF'):
    y0=np.zeros(87); y0[7:].reshape(5,16)[:,:2]=density(s)
    times=np.unique(np.r_[np.linspace(0,24,points),2.]); tt=[]; yy=[]
    for start,end,on in [(0,2,True),(2,24,False)]:
        sol=solve_ivp(make_rhs(theta,s,on),(start,end),y0,method=method,
                      rtol=1e-9 if tight else 2e-6,
                      atol=1e-11 if tight else 2e-9,dense_output=True)
        if not sol.success: raise RuntimeError(sol.message)
        ts=times[(times>=start)&(times<=end)] if start==0 else times[(times>start)&(times<=end)]
        tt.extend(ts); yy.extend(sol.sol(ts).T); y0=sol.y[:,-1]
    t=np.array(tt); y=np.array(yy); z=y[:,7:].reshape(-1,5,16)
    if y.min() < -2e-6:
        if not tight: return simulate(theta,s,points=points,tight=True,method=method)
        if method == 'BDF': return simulate(theta,s,points=points,tight=True,method='Radau')
        raise AssertionError(f'Negative scaled state {y.min()}')
    effect=z[:,:,14]/(4+z[:,:,14])
    SI=z[:,:,2:5]; SN=z[:,:,5:8]; EI=z[:,:,8:11]; EN=z[:,:,11:14]
    ab=VC*(y[:,0]+y[:,2])+VP*(y[:,1]+y[:,3])+(SI+SN+EI+EN).sum(axis=2)@CAP+y[:,5]
    payload=VC*y[:,0]+VP*y[:,1]+(SI+EI).sum(axis=2)@CAP+s.payload_volume*y[:,4]+z[:,:,14]@CELLVOL+y[:,6]
    administered=s.dose*np.minimum(t/2,1)
    mass=np.array([np.max(abs(ab-administered)),np.max(abs(payload-administered))])/max(s.dose,1.)
    bound=SI+SN
    receptor=z[:,:,:2]+np.stack([bound[:,:,0]+bound[:,:,2],bound[:,:,1]+bound[:,:,2]],axis=-1)
    return dict(t=t,y=y,z=z,effect=effect,tumor=effect[-1,0],healthy=effect[-1,1:]@F[1:],mass=mass,receptor=receptor)


def metrics(theta,s,tight=True):
    r=simulate(theta,s,tight=tight)
    e=r['effect'][-1]; z=r['z']; y=r['y']; t=r['t']
    rho=density(s); rates=10.**np.array(theta)
    ratio=np.divide(r['receptor'],rho[None],out=np.ones_like(r['receptor']),where=rho[None]>0)
    row=dict(tumor_pct=100*e[0],healthy_pct=100*r['healthy'],
             **{f'{n}_pct':100*v for n,v in zip(NAMES[1:],e[1:])},
             tumor_payload_uM=z[-1,0,14],targeted_delivery_uM=z[-1,0,15],
             tumor_surface_min=ratio[:,0,:].min(),H12_surface_min=ratio[:,4,:].min(),
             intact_AUC=np.trapezoid(y[:,0],t),total_Ab_AUC=np.trapezoid(y[:,0]+y[:,2],t),
             free_payload_AUC=np.trapezoid(y[:,4],t),mass_error=r['mass'].max(),
             KD1_uM=rates[1]/rates[0]*1e6,KD2_uM=rates[3]/rates[2]*1e6,
             bound_distance=min(min(x-lo,hi-x) for x,(lo,hi) in zip(theta,BOUNDS)),
             **dict(zip(RATES,rates)))
    for j,n in enumerate(NAMES):
        row[f'{n}_peak_uM']=z[:,j,14].max();row[f'{n}_AUC_uMh']=np.trapezoid(z[:,j,14],t)
    si=z[-1,:,2:5];sn=z[-1,:,5:8]
    for i in range(2):
        den=r['receptor'][-1,0,i]
        row[f'T_RO{i+1}_intact']=(si[0,i]+si[0,2])/den if den else 0.
        row[f'T_RO{i+1}_naked']=(sn[0,i]+sn[0,2])/den if den else 0.
    return row


### 3. Verification before optimisation

No dose; no targets; zero permeability; no spontaneous naked antibody; two mass balances; receptor accounting; full target relabelling; total-antibody invariance; fast-processing limit; BDF/Radau agreement. An independent reduced intact-only implementation checks v2-equivalent trajectories beyond the two calibration anchors. These checks establish numerical consistency, not biological validity.

In [ ]:
def validate():
    checks={}; x=np.array([4.,0.,4.,0.]);s=Scenario()
    r=simulate(x,s,tight=True)
    checks['Figure 6 uptake error uM']=abs(r['z'][-1,0,14]-280.)
    assert checks['Figure 6 uptake error uM']<.001
    f4=simulate(np.log10([4.6e5,3.4e-4,4.6e5,3.4e-4]),tight=True)
    assert abs(f4['z'][-1,0,14]-56)<.001
    zero=simulate(x,replace(s,dose=0),tight=True)
    np.testing.assert_allclose(zero['z'][:,:,:2],np.broadcast_to(density(s),zero['z'][:,:,:2].shape),atol=1e-10)
    assert np.max(abs(zero['z'][:,:,2:]))<1e-10
    assert np.max(abs(r['z'][:,:,5:14]))<1e-10
    for sc in [replace(s,half=48),replace(s,half=48,permeability=.1),
               replace(s,half=48,permeability=.1,tau=.25,recycle=.3),
               replace(s,ki=(.2,4,2))]:
        a=simulate(x,sc,tight=True); assert a['mass'].max()<1e-7
        clean=simulate(x,replace(sc,half=np.inf),tight=True)
        # Scaled states, not vacuous absolute tolerance on molar concentrations.
        total=lambda r:np.c_[r['y'][:,:2]+r['y'][:,2:4],
             (r['z'][:,:,2:5]+r['z'][:,:,5:8]).reshape(len(r['t']),-1),
             (r['z'][:,:,8:11]+r['z'][:,:,11:14]).reshape(len(r['t']),-1)]
        err=np.max(abs(total(a)-total(clean)))
        np.testing.assert_allclose(total(a),total(clean),rtol=1e-7,atol=1e-8)
        checks['max total-antibody invariance scaled error']=max(err,checks.get('max total-antibody invariance scaled error',0))
        checks['max relative mass error']=max(a['mass'].max(),checks.get('max relative mass error',0))
    leak=simulate(x,replace(s,half=48),tight=True)
    assert np.max(abs(leak['z'][:,1,14]))<1e-12
    notarget=simulate(x,replace(s,rho=(0,0),half=48,permeability=.1),tight=True)
    assert np.max(abs(notarget['z'][:,:,15]))<1e-12
    assert notarget['z'][-1,1,14]>0
    # Full target relabeling includes densities, free turnover, and bound trafficking.
    a=Scenario(rho=(10,.5),ki=(.2,4,2),k0=(.3,3),half=96,permeability=.01,tau=.25,recycle=.3)
    xx=np.array([3,-1,4,0.])
    ra=simulate(xx,a,tight=True)
    rb=simulate(xx[[2,3,0,1]],replace(a,rho=a.rho[::-1],ki=(4,.2,2),k0=a.k0[::-1]),tight=True)
    np.testing.assert_allclose(ra['effect'][-1],rb['effect'][-1,[0,1,3,2,4]],rtol=1e-7,atol=1e-8)
    # Very fast degradation without recycling tends to immediate release.
    fast=simulate(x,replace(s,tau=1e-5,process_share=0),tight=True)
    checks['fast-processing endpoint error']=abs(fast['tumor']-r['tumor'])
    assert checks['fast-processing endpoint error']<1e-6
    alt=simulate(x,replace(s,half=48,permeability=.1,tau=.25,recycle=.3),tight=True,method='Radau')
    bdf=simulate(x,replace(s,half=48,permeability=.1,tau=.25,recycle=.3),tight=True)
    checks['Radau-BDF endpoint error']=np.max(abs(alt['effect'][-1]-bdf['effect'][-1]))
    assert checks['Radau-BDF endpoint error']<1e-7
    # Receptor accounting is checked at a non-equilibrium state (all transfers cancel).
    yy=bdf['y'][20]; zz=yy[7:].reshape(5,16)
    sc=replace(s,half=48,permeability=.1,tau=.25,recycle=.3)
    dz=make_rhs(x,sc,False)(5,yy)[7:].reshape(5,16)
    kg=(1-sc.recycle)*(1-sc.process_share)/sc.tau
    for i in range(2):
        actual=dz[:,i]+dz[:,2+i]+dz[:,4]+dz[:,5+i]+dz[:,7]+dz[:,8+i]+dz[:,10]+dz[:,11+i]+dz[:,13]
        expected=sc.k0[i]*(density(sc)[:,i]-zz[:,i])-kg*(zz[:,8+i]+zz[:,10]+zz[:,11+i]+zz[:,13])
        np.testing.assert_allclose(actual,expected,atol=1e-7)
    return checks

validation=validate()
display(pd.Series(validation,name='Numerical verification'))
def legacy_v2(theta, points=97):
    """Independent intact-only v2 equations, frozen calibration, 4 populations."""
    a,b,c,d=10.**np.asarray(theta)*3600; q1=b*np.sqrt(c/a);q2=d/np.sqrt(c/a)
    l=2.4e13*1e-10*3600; ids=[0,2,3,4]; rho=density(Scenario())[ids]
    y0=np.zeros(27);y0[3:].reshape(4,6)[:,:2]=rho
    times=np.unique(np.r_[np.linspace(0,24,points),2]); yy=[]
    def rhs(on):
        def f(t,y):
            z=y[3:].reshape(4,6);o=np.zeros_like(y);dz=o[3:].reshape(4,6)
            r1,r2,s1,s2,du=z[:,:5].T
            v1=a*y[0]*1e-6*r1-b*s1;v2=c*y[0]*1e-6*r2-d*s2
            w1=l*s2*r1-q1*du;w2=l*s1*r2-q2*du
            dz[:,0]=2*(rho[:,0]-r1)-v1-w1;dz[:,1]=2*(rho[:,1]-r2)-v2-w2
            dz[:,2]=v1-w2-2*s1;dz[:,3]=v2-w1-2*s2;dz[:,4]=w1+w2-2*du
            dz[:,5]=GAIN*2*(s1+s2+du)
            dist=QA*(y[0]-y[1]);o[0]=((2 if on else 0)-CLA*y[0]-dist-CAP[ids]@(v1+v2))/VC
            o[1]=dist/VP;o[2]=CLA*y[0]
            return o
        return f
    for lo,hi,on in [(0,2,True),(2,24,False)]:
        sol=solve_ivp(rhs(on),(lo,hi),y0,method='BDF',rtol=1e-9,atol=1e-11,dense_output=True)
        assert sol.success
        ts=times[(times>=lo)&(times<=hi)] if lo==0 else times[(times>lo)&(times<=hi)]
        yy.extend(sol.sol(ts).T);y0=sol.y[:,-1]
    return np.array(yy)

legacy_errors=[]
for x in [[4,0,4,0],[2,-1.12,2.1,-1],[5,-3,4,-2]]:
    old=legacy_v2(x);new=simulate(x,tight=True)
    mapped=np.concatenate([new['y'][:,:2],new['z'][:,[0,2,3,4],:][:,:,[0,1,2,3,4,14]].reshape(len(old),-1)],axis=1)
    reference=np.c_[old[:,:2],old[:,3:]]
    np.testing.assert_allclose(mapped,reference,rtol=2e-7,atol=2e-7)
    legacy_errors.append(np.max(abs(mapped-reference)))
validation['max v2 trajectory error (scaled)']=max(legacy_errors)
print('Independent v2 trajectory equivalence passed:',max(legacy_errors))


In [ ]:
published=[]
for label,x,s in [
 ('Figure 4 bsADC',np.log10([4.6e5,3.4e-4,4.6e5,3.4e-4]),Scenario()),
 ('Figure 6 bsADC',[4,0,4,0],Scenario()),
 ('Figure 8 bsADC',[4,0,np.log10(5e4),0],Scenario(rho=(10,.5)))]:
    published.append(dict(candidate=label,**metrics(x,s)))
published=pd.DataFrame(published)
display(published[['candidate','tumor_payload_uM','tumor_pct','healthy_pct','H0_pct','H1_pct','H2_pct','H12_pct']].round(5))

### 4. Two-seed four-rate searches

Platypus NSGA-II (population 32, 192 evaluations per seed), then constrained SLSQP (55 iterations maximum). Seeds 8407 and 2026. Bounds match v2: kon 10²–10⁷ M⁻¹s⁻¹, koff 10⁻⁵–10² s⁻¹. Every selected candidate is reintegrated tightly. Baseline design is injected into each upgrade search; no favourable mechanism parameter is optimised. Search logs include local termination, all evaluated points, seed disagreement, and distance from bounds. Results are **best found feasible candidates**, not certified global optima. An empty search result does not prove mathematical infeasibility.

B1 varies dual internalisation and brackets slow T1 / fast T2 at equal and Figure 8 densities. B3 permits shedding but zero membrane exchange. B4 adds bidirectional exchange, including a **stable-linker exchange control** to separate payload efflux from linker shedding. B5 combines mechanisms. Unequal-density comparisons use their own B0 comparator.

Feasibility acceptance allows 10⁻⁸ in effect fraction around 0.0095 for numerical error; the strict 0.01 threshold remains enforced. Screening integrations with negative-state excursions are retried at tighter tolerances, then with Radau if necessary. Completed scenarios are checkpointed; `REUSE_CHECKPOINTS=False` is the default for a fresh run. Enable reuse only for an unchanged interrupted run.

In [ ]:
SEEDS=(8407,2026)
SEARCH_EVALUATIONS=192
WORKERS=4
REUSE_CHECKPOINTS=False  # True only to resume an unchanged interrupted run
CHECKPOINT=Path("results_v3/checkpoints");CHECKPOINT.mkdir(parents=True,exist_ok=True)
import hashlib
from joblib import Parallel, delayed, parallel_config

def optimize_case(name,s,starts=(),bounds=BOUNDS):
    key=hashlib.sha256(json.dumps([name,asdict(s),list(map(list,starts)),bounds,SEEDS,SEARCH_EVALUATIONS]).encode()).hexdigest()[:16]
    path=CHECKPOINT/f'{name}_{key}.json'
    if REUSE_CHECKPOINTS and path.exists():return json.loads(path.read_text())
    runs=[]
    for seed in SEEDS:
        random.seed(seed); cache={}; records=[]
        def ev(x):
            key=tuple(map(float,x))
            if key not in cache:
                r=simulate(x,s,points=3)
                cache[key]=(100*r['tumor'],100*r['healthy'])
                records.append([*key,*cache[key]])
            return cache[key]
        p=Problem(4,2,1);p.types[:]=[Real(*b) for b in bounds];p.constraints[:]='<=0'
        def objective(x):
            e,h=ev(x);return [-e,h],[h-100*LIMIT]
        p.function=objective
        init=[]
        for x in list(starts)+[np.log10([on,off,on,off]) for on,off in itertools.product([1e2,1e4,1e6],[.001,.1,1.,100.])]:
            a=Solution(p);a.variables[:]=np.clip(x,np.array(bounds)[:,0],np.array(bounds)[:,1]);init.append(a)
        alg=NSGAII(p,population_size=32,generator=InjectedPopulation(init));alg.run(SEARCH_EVALUATIONS)
        feasible=sorted([r for r in records if r[-1]<=100*LIMIT],key=lambda r:-r[-2])
        # Even with no feasible population, minimize toxicity before declaring no candidate found.
        x0=np.array(feasible[0][:4] if feasible else min(records,key=lambda r:r[-1])[:4])
        fit=minimize(lambda x:-ev(x)[0],x0,method='SLSQP',bounds=bounds,
                     constraints=[dict(type='ineq',fun=lambda x:100*(LIMIT-2e-7)-ev(x)[1])],
                     options=dict(maxiter=55,ftol=2e-6,eps=1e-3))
        ev(fit.x)
        best=None
        for row in sorted([r for r in records if r[-1]<=100*LIMIT+2e-5],key=lambda r:-r[-2])[:12]:
            m=metrics(row[:4],s)
            if m['healthy_pct']<=100*LIMIT+1e-6:
                best=dict(theta=row[:4],metrics=m);break
        runs.append(dict(seed=seed,best=best,nfe=len(records),local_success=bool(fit.success),
                         local_message=str(fit.message),evaluations=records))
    candidates=[r['best'] for r in runs if r['best']]
    best=max(candidates,key=lambda b:b['metrics']['tumor_pct']) if candidates else None
    result=dict(name=name,scenario=asdict(s),best=best,runs=runs)
    path.write_text(json.dumps(result))
    print(f'Completed {name}: '+(f"{best['metrics']['tumor_pct']:.3f}% tumour" if best else 'no feasible candidate found'),flush=True)
    return result

def parallel_cases(tasks):
    with parallel_config(backend='loky',inner_max_num_threads=1):
        return Parallel(n_jobs=WORKERS)(delayed(optimize_case)(*task) for task in tasks)


In [ ]:
# Every scenario shares physiology, dose, potency, scoring time and rate bounds.
baselines=parallel_cases([
    ('B0_equal',Scenario(),([2,-1.12,2,-1.12],)),
    ('B0_unequal',Scenario(rho=(10,.5)),([2,-1,3,0],))])
baseline_designs={r['name']:r['best']['theta'] for r in baselines}
assert all(r['best'] is not None for r in baselines)
display(pd.DataFrame([dict(case=r['name'],**r['best']['metrics']) for r in baselines])[
    ['case','tumor_pct','healthy_pct','H12_pct','KD1_uM','KD2_uM']].round(4))



In [ ]:
scenarios={}
for a in [.2,.5,1,2,5]:scenarios[f'B1_dual_{a:g}x']=Scenario(ki=(2,2,2*a))
for density_name,rho in [('equal',(1,1)),('unequal',(10,.5))]:
    for dual in [.2,2,4]:scenarios[f'B1_slowT1_{density_name}_dual{dual:g}']=Scenario(rho=rho,ki=(.2,4,dual))
for half in [48,96,168]:
    scenarios[f'B3_shed_{half}h']=Scenario(half=half)
    scenarios[f'B4_exchange_{half}h']=Scenario(half=half,permeability=.1)
# Stable-linker exchange controls isolate efflux/redistribution from cleavage.
scenarios['B4_stable_exchange']=Scenario(permeability=.1)
for perm in [.01,1.]:scenarios[f'B4_96h_perm{perm:g}']=Scenario(half=96,permeability=perm)
scenarios['B5_dual_fast']=Scenario(ki=(2,2,10),half=96,permeability=.1)
scenarios['B5_all']=Scenario(ki=(.2,4,2),half=96,permeability=.1,tau=.25,recycle=.3)

def baseline_for(s):return baseline_designs['B0_equal' if s.rho==(1,1) else 'B0_unequal']
started=time.perf_counter()
upgrades=parallel_cases([(name,s,(baseline_for(s),)) for name,s in scenarios.items()])
results={r['name']:r for r in baselines+upgrades}
comparison_rows=[]
for name,s in scenarios.items():
    fixed=metrics(baseline_for(s),s)
    best=results[name]['best']
    row=dict(case=name,fixed_tumor_pct=fixed['tumor_pct'],fixed_healthy_pct=fixed['healthy_pct'],
             status='feasible candidate found' if best else 'none found; not proof of infeasibility')
    if best:row.update(best['metrics'])
    comparison_rows.append(row)
comparison=pd.DataFrame(comparison_rows)
display(comparison[['case','fixed_tumor_pct','fixed_healthy_pct','tumor_pct','healthy_pct','H0_pct','H12_pct','bound_distance']].round(4))
print(f'{len(upgrades)} upgrade scenarios completed in {(time.perf_counter()-started)/60:.1f} minutes')


### 5. Recycling, payload-PK and binding-closure sensitivity

B2 is a focused fixed-antibody sensitivity module (six residence/recycling scenarios), not a full optimisation campaign. The payload grid varies three independent assumptions without calibration. Retuning the asymmetric case under the uncoupled binding convention tests whether its selected affinities depend on closure. Surface-association sensitivity is a fixed-design diagnostic. All individual healthy populations, peak concentrations, sampled AUC, productive delivery, occupancy and receptor abundance are retained in tables.

With unequal solution on-rates and fixed equal surface association rates, the uncoupled convention need not satisfy detailed balance. Treat it as a structural stress test, not an equally established physical model or a faithful implementation of v2's `sengers` option.

In [ ]:
# Focused B2 recycling/residence sensitivity: fixed baseline antibody, no retuning.
x=baseline_designs['B0_equal']; recycle_rows=[]
for tau,f in itertools.product([.25,1.],[0.,.3,.7]):
    s=Scenario(tau=tau,recycle=f)
    recycle_rows.append(dict(tau_h=tau,recycling_fraction=f,**metrics(x,s)))
recycling=pd.DataFrame(recycle_rows)
display(recycling[['tau_h','recycling_fraction','tumor_pct','healthy_pct','targeted_delivery_uM','T_RO1_naked']].round(5))
# All rates below are hypotheses, not payload-specific estimates or uncertainty intervals.
pk_rows=[]
for half,perm,clearance_rate in itertools.product([48,96,168],[.01,.1,1.],[.01,.1,1.]):
    s=Scenario(half=half,permeability=perm,payload_clearance=52.8*clearance_rate)
    pk_rows.append(dict(half_h=half,permeability_h=perm,clearance_h=clearance_rate,**metrics(x,s)))
pk_sensitivity=pd.DataFrame(pk_rows)
# Retest AND reoptimise an asymmetric case under the uncoupled closure.
closure_name='B1_slowT1_unequal_dual2'
s=scenarios[closure_name];old=results[closure_name]['best']['theta']
uncoupled=replace(s,closure='uncoupled')
closure_result=optimize_case('uncoupled_asymmetric',uncoupled,(old,baseline_for(s)))
closure_table=pd.DataFrame([
 dict(case='primary closure, selected',**metrics(old,s)),
 dict(case='same antibody, uncoupled',**metrics(old,uncoupled)),
 *([dict(case='uncoupled, reoptimised',**closure_result['best']['metrics'])] if closure_result['best'] else [])])
display(closure_table[['case','tumor_pct','healthy_pct','KD1_uM','KD2_uM']].round(5))
surface_rows=[]
for factor in [.1,1,10]:
    surface_rows.append(dict(surface_factor=factor,**metrics(old,replace(s,surface_factor=factor))))
display(pd.DataFrame(surface_rows)[['surface_factor','tumor_pct','healthy_pct']].round(5))
# Numerical stability of every reported selected design, feasibility and accounting.
verification=[]
for name,r in results.items():
    if r['best'] is None:continue
    s=Scenario(**r['scenario']);theta=r['best']['theta']
    tight=simulate(theta,s,tight=True);normal=simulate(theta,s)
    err=np.max(abs(tight['effect'][-1]-normal['effect'][-1]))
    assert err<2e-5 and tight['mass'].max()<1e-6 and tight['healthy']<.01
    verification.append(dict(case=name,tolerance_effect_error=err,mass_error=tight['mass'].max(),
        seed_spread_pp=np.ptp([a['best']['metrics']['tumor_pct'] for a in r['runs'] if a['best']]),
        local_successes=sum(a['local_success'] for a in r['runs'])))
verification=pd.DataFrame(verification)
display(verification.round(8))


### 6. Results

Shared plotting scales allow comparison. The Figure 5-like landscapes restrict both arms to identical rates and therefore do not establish a four-dimensional optimum. The original plotting domain and broader search domain are evaluated separately. Time-course peaks/AUC use the displayed sampling grid (97 points, plus infusion end).

In [ ]:
OUT=Path('results_v3');OUT.mkdir(exist_ok=True)
plt.rcParams.update({'figure.dpi':110,'axes.spines.top':False,'axes.spines.right':False,'font.size':10})
def show(fig,name):
    fig.savefig(OUT/f'{name}.png',dpi=170,bbox_inches='tight');plt.show();plt.close(fig)

fig,axes=plt.subplots(1,2,figsize=(11,4),layout='constrained')
alpha=np.array([.2,.5,1,2,5]);a=comparison.set_index('case').loc[[f'B1_dual_{v:g}x' for v in alpha]]
for ax,kind in zip(axes,['tumor','healthy']):
    ax.plot(alpha,a[f'fixed_{kind}_pct'],'o--',label='Baseline antibody')
    ax.plot(alpha,a[f'{kind}_pct'],'o-',label='Reoptimised')
    ax.set(xscale='log',xlabel='Dual internalisation multiplier',ylabel=f'{kind.title()} effect (%)')
axes[1].axhline(.95,color='black',ls=':',label='0.95% constraint');axes[0].legend();axes[1].legend()
show(fig,'trafficking')

keys=['B0_equal','B3_shed_96h','B4_stable_exchange','B4_exchange_96h','B5_all']
fig,axes=plt.subplots(1,2,figsize=(12,4),layout='constrained')
xx=np.arange(len(keys)); rows=[results[k]['best']['metrics'] if results[k]['best'] else {} for k in keys]
axes[0].bar(xx,[r.get('tumor_pct',np.nan) for r in rows],color='#336c9b')
axes[0].set(ylabel='Selected tumour effect (%)',ylim=(0,100))
bottom=np.zeros(len(keys))
for n,w in zip(NAMES[1:],F[1:]):
    vals=np.array([r.get(f'{n}_pct',np.nan)*w for r in rows]);axes[1].bar(xx,vals,bottom=bottom,label=n);bottom+=vals
axes[1].axhline(.95,color='black',ls=':');axes[1].set(ylabel='Weighted healthy effect (percentage points)');axes[1].legend(ncol=2)
for ax in axes:ax.set_xticks(xx,['Baseline','Shedding only','Stable + exchange','Shedding + exchange','Combined'],rotation=20,ha='right')
show(fig,'stage_comparison')



In [ ]:
# Figure 5-like symmetric slice. Four-dimensional searches above remain distinct.
grid_axes={'paper':(np.linspace(4,7,9),np.linspace(-3,0,9)),
           'extended':(np.linspace(2,7,13),np.linspace(-5,2,15))}
grid_cases={'B0':Scenario(),'B1_fast':Scenario(ki=(2,2,10)),
            'B4_stable':Scenario(permeability=.1),'B4_96h':Scenario(half=96,permeability=.1)}
def grid_eval(arg):
    name,domain,on,off,s=arg;r=simulate([on,off,on,off],s,points=3)
    return dict(case=name,domain=domain,log_on=on,log_off=off,tumor_pct=100*r['tumor'],healthy_pct=100*r['healthy'])
tasks=[(name,dom,on,off,s) for name,s in grid_cases.items() for dom,(ons,offs) in grid_axes.items() for off in offs for on in ons]
with parallel_config(backend='loky',inner_max_num_threads=1):
    grid=pd.DataFrame(Parallel(n_jobs=WORKERS)(delayed(grid_eval)(t) for t in tasks))
fig,axes=plt.subplots(2,4,figsize=(14,6),layout='constrained')
for col,name in enumerate(grid_cases):
    g=grid[(grid.case==name)&(grid.domain=='extended')];ons,offs=grid_axes['extended']
    for row,var in enumerate(['tumor_pct','healthy_pct']):
        vals=g.pivot(index='log_off',columns='log_on',values=var).values
        ax=axes[row,col];im=ax.pcolormesh(ons,offs,vals,shading='nearest',cmap='viridis' if row==0 else 'magma',vmin=0,vmax=100 if row==0 else 5)
        h=g.pivot(index='log_off',columns='log_on',values='healthy_pct').values
        if h.min()<.95<h.max():ax.contour(ons,offs,h,levels=[.95],colors='cyan',linewidths=1)
        ax.set(title=name if row==0 else '',xlabel='log10 kon (M⁻¹ s⁻¹)',ylabel='log10 koff (s⁻¹)')
    fig.colorbar(im,ax=axes[1,col],label='Healthy effect (%)',extend='max')
fig.colorbar(axes[0,0].collections[0],ax=axes[0,:],label='Tumour effect (%)')
fig.suptitle('Symmetric slices; cyan = 0.95% healthy effect (not a 4D optimum)')
show(fig,'binding_landscapes')
display(grid.groupby(['domain','case']).agg(min_healthy_pct=('healthy_pct','min'),max_tumor_pct=('tumor_pct','max')).round(4))



In [ ]:
fig,axes=plt.subplots(1,3,figsize=(12,3.6),layout='constrained')
for ax,half in zip(axes,[48,96,168]):
    z=pk_sensitivity[pk_sensitivity.half_h==half].pivot(index='permeability_h',columns='clearance_h',values='healthy_pct')
    im=ax.imshow(z,origin='lower',aspect='auto',vmin=0,vmax=max(1,pk_sensitivity.healthy_pct.max()),cmap='magma')
    for i in range(3):
        for j in range(3):ax.text(j,i,f'{z.iloc[i,j]:.2f}',ha='center',va='center',color='white' if z.iloc[i,j]<2 else 'black')
    ax.set(xticks=range(3),xticklabels=z.columns,yticks=range(3),yticklabels=z.index,xlabel='Payload clearance / volume (h⁻¹)',ylabel='PS / cell volume (h⁻¹)',title=f'Linker half-life {half} h')
fig.colorbar(im,ax=axes,label='Healthy effect (%), baseline antibody fixed')
show(fig,'payload_sensitivity')

# Selected trajectories expose concentration, historical delivery, and receptor changes.
fig,axes=plt.subplots(1,3,figsize=(12,3.7),layout='constrained')
for name in ['B0_equal','B4_exchange_96h','B5_all']:
    b=results[name]['best']
    if b is None:continue
    r=simulate(b['theta'],Scenario(**results[name]['scenario']),points=241,tight=True)
    axes[0].plot(r['t'],r['z'][:,0,14],label=name)
    axes[1].plot(r['t'],r['z'][:,1,14],label=name)
    axes[2].plot(r['t'],r['receptor'][:,0,0],label=name)
for ax,title,y in zip(axes,['Tumour active payload','Target-negative healthy payload','Tumour surface T1'],['µM','µM','Fraction of baseline T1']):ax.set(title=title,xlabel='Time (h)',ylabel=y)
axes[0].legend(fontsize=8);show(fig,'selected_trajectories')


In [ ]:
# Reproducible numerical outputs; not inputs required to execute this notebook.
comparison.to_csv(OUT/'stage_comparison.csv',index=False)
published.to_csv(OUT/'published_comparators.csv',index=False)
recycling.to_csv(OUT/'recycling_sensitivity.csv',index=False)
pk_sensitivity.to_csv(OUT/'payload_sensitivity.csv',index=False)
closure_table.to_csv(OUT/'closure_sensitivity.csv',index=False)
verification.to_csv(OUT/'verification.csv',index=False)
grid.to_csv(OUT/'symmetric_landscapes.csv',index=False)
(OUT/'search_results.json').write_text(json.dumps(dict(results=results,closure=closure_result),indent=2))
(OUT/'validation.json').write_text(json.dumps(validation,indent=2))
print('Saved tables, figures and full search records to',OUT.resolve())

### Additional result checks

Selected rate table, target-swap landscapes, exposure sampling convergence, and structural sensitivities. All use the same frozen controls.

In [ ]:
# Parameter selection, subgroup effects, and evidence of search convergence.
selected_rates=pd.DataFrame([dict(case=name,**r['best']['metrics']) for name,r in results.items() if r['best']])
display(selected_rates[['case',*RATES,'KD1_uM','KD2_uM','bound_distance']].round(6))
selected_rates.to_csv(OUT/'selected_designs.csv',index=False)

# Test loss of target-label symmetry on an equal-density off-rate slice.
offs=np.linspace(-3,1,11)
def asym_eval(arg):
    name,off1,off2=arg;s=Scenario() if name=='equal trafficking' else Scenario(ki=(.2,4,2))
    r=simulate([3,off1,3,off2],s,points=3)
    return dict(case=name,off1=off1,off2=off2,tumor_pct=100*r['tumor'],healthy_pct=100*r['healthy'])
tasks=[(name,i,j) for name in ['equal trafficking','slow T1 / fast T2'] for j in offs for i in offs]
with parallel_config(backend='loky',inner_max_num_threads=1):
    asymmetry=pd.DataFrame(Parallel(n_jobs=WORKERS)(delayed(asym_eval)(t) for t in tasks))
fig,axes=plt.subplots(2,2,figsize=(9,7),layout='constrained')
symmetry_diagnostics=[]
for col,name in enumerate(['equal trafficking','slow T1 / fast T2']):
    for row,key in enumerate(['tumor_pct','healthy_pct']):
        mat=asymmetry[asymmetry.case==name].pivot(index='off2',columns='off1',values=key).values
        ax=axes[row,col];im=ax.pcolormesh(offs,offs,mat,shading='nearest',cmap='viridis' if row==0 else 'magma',vmin=0,vmax=100 if row==0 else 8)
        ax.set(title=name if row==0 else '',xlabel='log10 koff1 (s⁻¹)',ylabel='log10 koff2 (s⁻¹)')
        fig.colorbar(im,ax=ax,label=key.replace('_',' '),extend='max' if row else 'neither')
        symmetry_diagnostics.append(dict(case=name,metric=key,max_swap_difference_pp=np.max(abs(mat-mat.T))))
show(fig,'asymmetric_trafficking')
display(pd.DataFrame(symmetry_diagnostics))
asymmetry.to_csv(OUT/'asymmetric_landscape.csv',index=False)

# Sampling convergence of exposure summaries (not a fitted PD model).
exposure_checks=[]
for name in ['B4_exchange_96h','B5_all']:
    best=results[name]['best']
    if best is None:continue
    sc=Scenario(**results[name]['scenario']);a=simulate(best['theta'],sc,points=97,tight=True);b=simulate(best['theta'],sc,points=769,tight=True)
    au=np.trapezoid(a['z'][:,:,14],a['t'],axis=0);bu=np.trapezoid(b['z'][:,:,14],b['t'],axis=0)
    exposure_checks.append(dict(case=name,max_AUC_relative_difference=np.max(abs(au-bu)/np.maximum(bu,1e-12)),
        max_peak_relative_difference=np.max(abs(a['z'][:,:,14].max(axis=0)-b['z'][:,:,14].max(axis=0))/np.maximum(b['z'][:,:,14].max(axis=0),1e-12))))
display(pd.DataFrame(exposure_checks))
print(f'H0-only necessary concentration ceiling: {4*LIMIT/(.79-LIMIT):.5f} µM at the 0.95% constraint; other healthy populations lower the available budget.')
peak_rows=[]
for name,result in results.items():
    if result['best'] is None:continue
    r=simulate(result['best']['theta'],Scenario(**result['scenario']),points=385,tight=True)
    healthy=100*(r['effect'][:,1:]@F[1:]);i=int(np.argmax(healthy))
    peak_rows.append(dict(case=name,healthy_24h_pct=healthy[-1],healthy_peak_pct=healthy[i],peak_time_h=r['t'][i]))
peak_effects=pd.DataFrame(peak_rows)
display(peak_effects.round(6))
peak_effects.to_csv(OUT/'healthy_peak_effects.csv',index=False)


In [ ]:
# Structural / branching checks use fixed antibodies, not tuned mechanism parameters.
structural_rows=[]
x=baseline_designs['B0_equal']
for vol in [11.3,52.8]:
    sc=Scenario(half=96,permeability=.1,payload_volume=vol,payload_clearance=.1*vol)
    structural_rows.append(dict(check='payload distribution volume',value=vol,**metrics(x,sc)))
for share in [0.,.5,1.]:
    sc=Scenario(tau=.25,recycle=.3,process_share=share)
    structural_rows.append(dict(check='productive processing share',value=share,**metrics(x,sc)))
structural_sensitivity=pd.DataFrame(structural_rows)
display(structural_sensitivity[['check','value','tumor_pct','healthy_pct','H0_pct','H12_pct','T_RO1_naked']].round(5))
structural_sensitivity.to_csv(OUT/'structural_sensitivity.csv',index=False)
small_pool=Scenario(half=96,permeability=.1,payload_volume=11.3,payload_clearance=1.13)
small_starts=[baseline_designs['B0_equal']]
if results['B4_exchange_96h']['best']:
    small_starts.append(results['B4_exchange_96h']['best']['theta'])
small_pool_result=optimize_case('B4_small_payload_pool',small_pool,small_starts)
(OUT/'small_pool_search.json').write_text(json.dumps(small_pool_result,indent=2))
if small_pool_result['best']:
    display(pd.DataFrame([dict(case='11.3 L pool, reoptimised',**small_pool_result['best']['metrics'])])[
        ['case','tumor_pct','healthy_pct','H0_pct','H12_pct','KD1_uM','KD2_uM','bound_distance']])
else:
    print('No feasible candidate found for the smaller payload pool; this is not a proof of infeasibility.')

if small_pool_result['best']:
    t=small_pool_result['best']['theta']
    a=simulate(t,small_pool,tight=True,points=385)
    b=simulate(t,small_pool,tight=True,method='Radau')
    assert np.max(abs(a['effect'][-1]-b['effect'][-1]))<1e-7
    assert a['mass'].max()<1e-6


### 7. Interpretation guardrails

- Compare fixed-design changes with reoptimised changes; improvement from redesign is distinct from improvement from biology.
- Faster dual uptake also acts on H12 and changes receptor abundance. Global <1% effect can conceal substantial H12 effect.
- With identical intact/naked kinetics, total antibody occupancy is invariant to cleavage. Loss of armed occupancy is not an independent increase in total receptor blockade.
- Exchange releases targeted intracellular payload back into the common pool even with a perfectly stable linker. Attribute changes to shedding only relative to the stable-exchange control.
- No interior “Goldilocks” optimum is presupposed. Boundary candidates, flat trade-offs and no feasible candidate found are all possible. Local termination and seed disagreement limit rate-specific claims.
- Emax of 24 h concentration can decrease after exposure falls. It is not a validated irreversible kill model; AUC and peaks are descriptive, without fitted PD conversion.
- New trafficking/PK assumptions are uncalibrated. No organ-specific toxicity, spatial bystander transport, heterogeneous tumours, repeated doses or robustness guarantee is inferred.

Mechanistic sources: [Stoddard et al.](https://doi.org/10.20935/AcadDrug8407), [Sengers et al.](https://pmc.ncbi.nlm.nih.gov/articles/PMC4968105/), [Lee et al. 2019](https://pmc.ncbi.nlm.nih.gov/articles/PMC6548581/), [Perez Bay et al.](https://pmc.ncbi.nlm.nih.gov/articles/PMC9978886/), [Glassman](https://www.mdpi.com/2072-6694/18/1/5). The cited studies motivate mechanisms, not the exploratory numerical rates above. Source discrepancies retained: Table 2 reverses Figure 4 panel labels; its Figure 8 on-rate implies stronger T2 affinity despite the prose saying reduced affinity. Supplement S3 also shows ~1.8% healthy effect near the reference point.